# 01 — EDA + Feature Engineering

**Structure:** quality checks -> feature engineering -> EDA on everything -> save.

Features are built *before* any EDA plots so every correlation and heatmap includes them.

**Input:** `data/raw/Sample Media Spend Data.csv`
**Output:** `data/processed/clean.parquet` (27 cols), `data/processed/eda_stats.json`

| Group | Features | Source |
|-------|---------|--------|
| Calendar | `is_q4`, `trend`, `sin_1`, `cos_1`, `sin_2`, `cos_2` | Derived from date |
| Holidays | `has_federal_holiday`, `is_black_friday_week`, `is_back_to_school`, `is_easter_week` | US calendar |
| Macro | `consumer_sentiment`, `unemployment_rate`, `cpi`, `personal_savings_rate` | FRED (free, no key) |

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from numpy.linalg import lstsq
from pandas.tseries.holiday import USFederalHolidayCalendar

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

RAW  = Path('../data/raw')
PROC = Path('../data/processed')
PROC.mkdir(exist_ok=True)

DATE_COL     = 'date'
DIV_COL      = 'Division'
TARGET_COL   = 'Sales'
CHANNEL_COLS = [
    'Paid_Views',
    'Google_Impressions',
    'Email_Impressions',
    'Facebook_Impressions',
    'Affiliate_Impressions',
]

print('Setup complete.')

## 1. Load & parse

In [ ]:
raw = pd.read_csv(RAW / 'Sample Media Spend Data.csv')
raw['date'] = pd.to_datetime(raw['Calendar_Week'])
raw = raw.drop(columns=['Calendar_Week', 'Overall_Views'])
raw = raw.sort_values([DIV_COL, DATE_COL]).reset_index(drop=True)

print(raw.shape)
print('Date range:', raw['date'].min(), '->', raw['date'].max())
print('Divisions :', sorted(raw[DIV_COL].unique()))
raw.head()

## 2. Quality checks

In [ ]:
print('Dtypes:\n', raw.dtypes)
print('\nMissing values:')
print(raw.isnull().sum())
raw.describe()

In [ ]:
weeks_per_div = raw.groupby(DIV_COL)['date'].nunique().sort_values()
print(weeks_per_div)
print('\nBalanced panel?', weeks_per_div.nunique() == 1)

---
## Feature Engineering
All features are built here, before any EDA, so every plot includes them.

## 3. Calendar features + Fourier seasonality terms

`week_of_year`, `month`, `quarter` are kept in `df` for EDA plots but are NOT
passed to the model. Instead we use **Fourier terms** (K=2 sin/cos pairs), which
capture a smooth seasonal curve rather than 52 independent week levels.

In [ ]:
df = raw.copy()

# Raw calendar — kept for EDA plots and Fourier computation
df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)
df['month']        = df['date'].dt.month
df['quarter']      = df['date'].dt.quarter
df['year']         = df['date'].dt.year
df['is_q4']        = (df['quarter'] == 4).astype(int)
df['trend']        = ((df['date'] - df['date'].min()).dt.days // 7).astype(int)

# Fourier seasonality terms (K=2 pairs at annual frequency = 52 weeks)
# sin_1/cos_1 = fundamental annual cycle (one peak per year)
# sin_2/cos_2 = semi-annual harmonic (two peaks per year: back-to-school + Q4)
K = 2
for k in range(1, K + 1):
    df[f'sin_{k}'] = np.sin(2 * np.pi * k * df['week_of_year'] / 52)
    df[f'cos_{k}'] = np.cos(2 * np.pi * k * df['week_of_year'] / 52)

FOURIER_COLS  = [f'sin_{k}' for k in range(1, K+1)] + [f'cos_{k}' for k in range(1, K+1)]
CALENDAR_COLS = ['is_q4', 'trend'] + FOURIER_COLS

print('CALENDAR_COLS (model features):', CALENDAR_COLS)
df[['date', 'week_of_year', 'is_q4', 'trend'] + FOURIER_COLS].drop_duplicates('date').head(8)

### What do Fourier terms capture?
K=1 captures the dominant annual shape. K=2 adds a semi-annual harmonic so
the model can distinguish back-to-school from Q4 as separate seasonal bumps.
The right panel below shows how well K=2 reconstructs the actual sales seasonality.

In [ ]:
weeks = np.arange(1, 53)
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

for k in range(1, K + 1):
    axes[0].plot(weeks, np.sin(2 * np.pi * k * weeks / 52), label=f'sin_{k}')
    axes[0].plot(weeks, np.cos(2 * np.pi * k * weeks / 52), label=f'cos_{k}', linestyle='--')
axes[0].axhline(0, color='black', linewidth=0.6)
axes[0].set_title(f'Fourier Basis Terms (K={K})')
axes[0].set_xlabel('Week of Year')
axes[0].legend(fontsize=9)

weekly_sales = df.groupby('week_of_year')[TARGET_COL].mean().sort_index()
X_f = np.column_stack([
    *[np.sin(2 * np.pi * k * weeks / 52) for k in range(1, K + 1)],
    *[np.cos(2 * np.pi * k * weeks / 52) for k in range(1, K + 1)],
    np.ones(52),
])
coefs, _, _, _ = lstsq(X_f, weekly_sales.values, rcond=None)
reconstructed  = X_f @ coefs

axes[1].plot(weeks, weekly_sales.values, alpha=0.7, label='Actual mean Sales')
axes[1].plot(weeks, reconstructed, linewidth=2.5, label=f'Fourier K={K} fit')
axes[1].set_title('Seasonality: Actual vs Fourier Reconstruction')
axes[1].set_xlabel('Week of Year')
axes[1].set_ylabel('Mean Sales')
axes[1].legend()

ss_res = ((weekly_sales.values - reconstructed) ** 2).sum()
ss_tot = ((weekly_sales.values - weekly_sales.mean()) ** 2).sum()
r2 = 1 - ss_res / ss_tot
print(f'Fourier K={K} explains {r2*100:.1f}% of variance in mean weekly Sales')

plt.tight_layout()
fig.savefig(PROC / 'fig_fourier_seasonality.png', bbox_inches='tight')

## 4. US Holiday flags

Holiday weeks cause sharp sales spikes the model would otherwise misattribute
to whichever media channel happened to be running that week.

In [ ]:
cal = USFederalHolidayCalendar()
holiday_dates = cal.holidays(start='2017-01-01', end='2020-12-31')
unique_dates  = df['date'].drop_duplicates().sort_values()

def week_has_holiday(week_start, holidays):
    week_end = week_start + pd.Timedelta(days=6)
    return int(((holidays >= week_start) & (holidays <= week_end)).any())

df['has_federal_holiday'] = df['date'].map(
    {d: week_has_holiday(d, holiday_dates) for d in unique_dates}
)

def black_friday_of(year):
    nov1 = pd.Timestamp(year, 11, 1)
    fourth_thu = nov1 + pd.Timedelta(days=(3 - nov1.weekday()) % 7) + pd.Timedelta(weeks=3)
    return fourth_thu + pd.Timedelta(days=1)

bf_dates = [black_friday_of(y) for y in df['year'].unique()]
df['is_black_friday_week'] = df['date'].map({
    d: int(any(d <= bf <= d + pd.Timedelta(days=6) for bf in bf_dates))
    for d in unique_dates
})

df['is_back_to_school'] = df['week_of_year'].between(31, 38).astype(int)

easter_dates = [pd.Timestamp('2018-04-01'), pd.Timestamp('2019-04-21'), pd.Timestamp('2020-04-12')]
df['is_easter_week'] = df['date'].map({
    d: int(any(d <= e <= d + pd.Timedelta(days=6) for e in easter_dates))
    for d in unique_dates
})

HOLIDAY_COLS = ['has_federal_holiday', 'is_black_friday_week', 'is_back_to_school', 'is_easter_week']
print('Holiday weeks flagged (out of', df['date'].nunique(), 'unique weeks):')
for col in HOLIDAY_COLS:
    print(f'  {col:<25} {int(df.drop_duplicates("date")[col].sum()):>3} weeks')

## 5. Macro features — FRED (free, no API key)

Four monthly series fetched from the Federal Reserve and forward-filled to weekly.

| FRED Series | Column | What it measures |
|-------------|--------|------------------|
| UMCSENT | `consumer_sentiment` | Univ. of Michigan sentiment index |
| UNRATE | `unemployment_rate` | US unemployment rate (%) |
| CPIAUCSL | `cpi` | Consumer Price Index — inflation |
| PSAVERT | `personal_savings_rate` | Personal savings rate (%) |

In [ ]:
def fetch_fred(series_id, col_name):
    url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}'
    try:
        fred = pd.read_csv(url)
        fred.columns = fred.columns.str.strip()
        date_col = next(c for c in fred.columns if 'date' in c.lower())
        fred = fred.rename(columns={date_col: 'date', series_id: col_name})
        fred['date'] = pd.to_datetime(fred['date'])
        fred = fred[fred[col_name] != '.'].copy()
        fred[col_name] = fred[col_name].astype(float)
        print(f'  {series_id}: {len(fred)} monthly obs fetched')
        return fred[['date', col_name]]
    except Exception as e:
        print(f'  {series_id}: fetch failed ({e}) -- using 0.0 fallback')
        return None

def merge_monthly(df_weekly, df_monthly, col):
    if df_monthly is None:
        df_weekly[col] = 0.0
        return df_weekly
    return pd.merge_asof(
        df_weekly.sort_values('date'),
        df_monthly.sort_values('date'),
        on='date', direction='backward'
    ).sort_values([DIV_COL, 'date']).reset_index(drop=True)

print('Fetching FRED series...')
umcsent = fetch_fred('UMCSENT',  'consumer_sentiment')
unrate  = fetch_fred('UNRATE',   'unemployment_rate')
cpi     = fetch_fred('CPIAUCSL', 'cpi')
psavert = fetch_fred('PSAVERT',  'personal_savings_rate')

df = merge_monthly(df, umcsent, 'consumer_sentiment')
df = merge_monthly(df, unrate,  'unemployment_rate')
df = merge_monthly(df, cpi,     'cpi')
df = merge_monthly(df, psavert, 'personal_savings_rate')

MACRO_COLS = ['consumer_sentiment', 'unemployment_rate', 'cpi', 'personal_savings_rate']

print('\nSample macro values (last 6 unique weeks):')
print(df[['date'] + MACRO_COLS].drop_duplicates('date').tail(6).to_string(index=False))

---
## EDA — all features included
Feature engineering complete. Every plot below uses the enriched `df`.

## 6. Sales time series — all 26 divisions (small multiples)

In [ ]:
divisions = sorted(df[DIV_COL].unique())
ncols = 6
nrows = -(-len(divisions) // ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(20, nrows * 3), sharex=True)
for ax, div in zip(axes.flat, divisions):
    sub = df[df[DIV_COL] == div]
    ax.plot(sub['date'], sub[TARGET_COL], linewidth=1)
    ax.set_title(f'Division {div}', fontsize=9)
    ax.tick_params(labelsize=7)
for ax in axes.flat[len(divisions):]:
    ax.set_visible(False)
fig.suptitle('Weekly Sales per Division', fontsize=13, y=1.01)
plt.tight_layout()
fig.savefig(PROC / 'fig_sales_small_multiples.png', bbox_inches='tight')

## 7. Seasonality — Sales by month and week-of-year

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

monthly = df.groupby('month')[TARGET_COL].mean()
axes[0].bar(monthly.index, monthly.values)
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'], rotation=45)
axes[0].set_title('Mean Sales by Month')
axes[0].set_ylabel('Mean Sales')

weekly_avg = df.groupby('week_of_year')[TARGET_COL].mean()
axes[1].plot(weekly_avg.index, weekly_avg.values, linewidth=1.5)
axes[1].set_title('Mean Sales by Week of Year')
axes[1].set_xlabel('Week of Year')
axes[1].set_ylabel('Mean Sales')

plt.tight_layout()
fig.savefig(PROC / 'fig_seasonality.png', bbox_inches='tight')

## 8. Holiday impact on Sales

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
titles = ['Federal Holiday', 'Black Friday Week', 'Back-to-School', 'Easter Week']
for ax, col, title in zip(axes, HOLIDAY_COLS, titles):
    groups = df.groupby(col)[TARGET_COL].mean()
    ax.bar(['Normal', 'Holiday'], [groups.get(0, 0), groups.get(1, 0)],
           color=['#636EFA', '#EF553B'])
    ax.set_title(title)
    ax.set_ylabel('Mean Sales')
    uplift = (groups.get(1, 0) / max(groups.get(0, 1), 1) - 1) * 100
    ax.set_xlabel(f'Uplift: {uplift:+.1f}%')
plt.suptitle('Mean Sales: Holiday vs Normal Weeks', fontsize=12)
plt.tight_layout()
fig.savefig(PROC / 'fig_holiday_impact.png', bbox_inches='tight')

## 9. Macro indicators vs total weekly Sales

In [ ]:
agg = df.groupby('date').agg(
    total_sales=(TARGET_COL, 'sum'),
    consumer_sentiment=('consumer_sentiment', 'first'),
    unemployment_rate=('unemployment_rate', 'first'),
    cpi=('cpi', 'first'),
    personal_savings_rate=('personal_savings_rate', 'first'),
).reset_index()

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, col, color in zip(axes, MACRO_COLS, ['#636EFA', '#EF553B', '#00CC96', '#AB63FA']):
    ax.scatter(agg[col], agg['total_sales'], alpha=0.5, s=20, color=color)
    ax.set_xlabel(col)
    ax.set_ylabel('Total Weekly Sales')
    ax.set_title(f'{col} vs Sales')
plt.tight_layout()
fig.savefig(PROC / 'fig_macro_vs_sales.png', bbox_inches='tight')

## 10. Channel spend distributions

In [ ]:
plot_cols = CHANNEL_COLS + ['Organic_Views']
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flat, plot_cols):
    ax.hist(df[col].dropna(), bins=50, edgecolor='none')
    ax.set_title(col, fontsize=10)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
for ax in axes.flat[len(plot_cols):]:
    ax.set_visible(False)
plt.suptitle('Channel Distributions (all divisions combined)', fontsize=12)
plt.tight_layout()
fig.savefig(PROC / 'fig_channel_distributions.png', bbox_inches='tight')

## 11. Zero-spend weeks per channel

In [ ]:
zero_pct = (df[CHANNEL_COLS] == 0).mean() * 100
print('Percentage of zero-spend weeks per channel:')
print(zero_pct.to_string())

## 12. Correlation heatmap — all features vs Sales

In [ ]:
corr_cols = CHANNEL_COLS + ['Organic_Views'] + CALENDAR_COLS + HOLIDAY_COLS + MACRO_COLS + [TARGET_COL]
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(18, 15))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.4, annot_kws={'size': 7}, ax=ax)
ax.set_title('Pearson Correlation -- All Features vs Sales', fontsize=13)
plt.tight_layout()
fig.savefig(PROC / 'fig_correlation_heatmap.png', bbox_inches='tight')

## 13. Feature importance — Pearson r with Sales (ranked)

In [ ]:
all_features = CHANNEL_COLS + ['Organic_Views'] + CALENDAR_COLS + HOLIDAY_COLS + MACRO_COLS
corr_with_sales = df[all_features + [TARGET_COL]].corr()[TARGET_COL].drop(TARGET_COL).sort_values()

fig, ax = plt.subplots(figsize=(8, 9))
colors = ['#d62728' if v < 0 else '#2ca02c' for v in corr_with_sales.values]
ax.barh(corr_with_sales.index, corr_with_sales.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title(f'Pearson r with {TARGET_COL} -- all features ranked')
ax.set_xlabel('Correlation')
plt.tight_layout()
fig.savefig(PROC / 'fig_feature_correlations.png', bbox_inches='tight')
print(corr_with_sales.to_string())

## 14. Spend vs Sales scatter — channels

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col in zip(axes.flat, CHANNEL_COLS + ['Organic_Views']):
    ax.scatter(df[col], df[TARGET_COL], alpha=0.15, s=10)
    ax.set_xlabel(col)
    ax.set_ylabel(TARGET_COL)
    ax.set_title(f'{col} vs {TARGET_COL}')
for ax in axes.flat[len(CHANNEL_COLS) + 1:]:
    ax.set_visible(False)
plt.suptitle('Channel Volume vs Sales', fontsize=12)
plt.tight_layout()
fig.savefig(PROC / 'fig_scatter_spend_vs_sales.png', bbox_inches='tight')

## 15. Rolling mean — stationarity check

In [ ]:
agg2 = df.groupby('date')[TARGET_COL].sum().reset_index()
agg2['rolling_12w'] = agg2[TARGET_COL].rolling(12, center=True).mean()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(agg2['date'], agg2[TARGET_COL], alpha=0.4, label='Weekly total')
ax.plot(agg2['date'], agg2['rolling_12w'], linewidth=2, label='12-week rolling mean')
ax.set_title('Total Sales -- All Divisions')
ax.set_xlabel('Date')
ax.set_ylabel('Sales')
ax.legend()
plt.tight_layout()
fig.savefig(PROC / 'fig_sales_rolling.png', bbox_inches='tight')

## 16. Confirm final column list before saving

In [ ]:
ALL_CONTROL_COLS = (
    ['Organic_Views']
    + CALENDAR_COLS
    + HOLIDAY_COLS
    + MACRO_COLS
)

missing = [c for c in ALL_CONTROL_COLS if c not in df.columns]
nulls   = df[ALL_CONTROL_COLS].isnull().sum()

print('Missing columns :', missing or 'none')
print('Null counts      :', 'none' if not nulls.any() else nulls[nulls > 0].to_dict())
print(f'Total df columns : {len(df.columns)}')
print('ALL_CONTROL_COLS :', ALL_CONTROL_COLS)

## 17. Save clean.parquet + eda_stats.json

In [ ]:
stats = {
    'n_rows': int(len(df)),
    'n_divisions': int(df[DIV_COL].nunique()),
    'date_min': str(df['date'].min().date()),
    'date_max': str(df['date'].max().date()),
    'channel_cols': CHANNEL_COLS,
    'control_cols': ALL_CONTROL_COLS,
    'target_col': TARGET_COL,
    'channel_means': df[CHANNEL_COLS].mean().round(1).to_dict(),
    'sales_mean': float(df[TARGET_COL].mean().round(1)),
    'sales_std': float(df[TARGET_COL].std().round(1)),
    'macro_available': bool(df['consumer_sentiment'].notna().all()),
}
with open(PROC / 'eda_stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

df.to_parquet(PROC / 'clean.parquet', index=False)
print(f'Saved {len(df):,} rows -> data/processed/clean.parquet')
print(f'{len(df.columns)} columns:', list(df.columns))